In [ ]:
!pip install opencv-python  # OpenCV for video processing
!pip install scikit-learn   # For KNN and other ML tools
!pip install torch torchvision opencv-python scikit-learn

In [ ]:
import os
import torch
import torchvision.transforms as transforms
import cv2
import numpy as np
from torchvision.models.video import r3d_18  # Import the 3D ResNet model
from sklearn.neighbors import NearestNeighbors

# Initialize the pre-trained 3D ResNet model
model = r3d_18(pretrained=True).eval()

# Move the model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Directory where videos are stored (including subfolders)
video_dir = '/kaggle/input/youtube-video-crafts'  # Replace with your Kaggle input video directory
unique_video_dir = '/kaggle/working/unique_videos-crafts/'  # Directory to save unique videos

# Create directory to save unique videos
if not os.path.exists(unique_video_dir):
    os.makedirs(unique_video_dir)

# Function to extract key frames from a video
def extract_key_frames(video_path, frame_count=32):
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    if not cap.isOpened():
        print(f"Error: Unable to open video file {video_path}. Skipping.")
        return frames
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        print(f"Error: No frames in video file {video_path}. Skipping.")
        return frames
    
    frame_indices = np.linspace(0, total_frames - 1, frame_count, dtype=int)

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        
        if ret:
            try:
                # Resize for 3D ResNet input
                frame_resized = cv2.resize(frame, (112, 112))
                frames.append(cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB))
            except Exception as e:
                print(f"Error processing frame {idx} from video {video_path}: {e}")
        else:
            print(f"Error reading frame {idx} from video {video_path}. Skipping this frame.")

    cap.release()
    return frames

# Function to get video embedding by averaging embeddings of key frames
def get_video_embedding(video_path):
    frames = extract_key_frames(video_path)

    # Check if we have enough frames
    if len(frames) < 16:
        print(f"Not enough frames in video {video_path}. Skipping.")
        return None  # Return None if not enough frames

    # Process frames in batches of 16
    with torch.no_grad():
        embeddings = []
        
        for i in range(0, len(frames), 16):
            batch_frames = frames[i:i + 16]
            if len(batch_frames) < 16:
                break  # Skip incomplete batch

            # Create tensor for the batch
            input_tensor = torch.zeros((1, 3, 16, 112, 112), device=device)
            for j, frame in enumerate(batch_frames):
                frame_tensor = torch.from_numpy(frame).permute(2, 0, 1).float()  # Change from [H, W, C] to [C, H, W]
                input_tensor[0, :, j, :, :] = frame_tensor  # Fill the input tensor with frames
            
            # Get embeddings using 3D ResNet
            batch_embeddings = model(input_tensor)  # Assuming the model returns embeddings directly
            embeddings.append(batch_embeddings.cpu().numpy().flatten())

        # Average the embeddings for all processed batches
        return np.mean(embeddings, axis=0)

# Function to recursively gather all video files from a directory and its subdirectories
def gather_video_files(directory):
    video_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith('.mp4'):  # Adjust if using other video formats
                video_files.append(os.path.join(root, file))
    return video_files

# Gather all video file paths (including subdirectories)
video_files = gather_video_files(video_dir)

# Extract embeddings for all videos
print("Extracting embeddings for all videos...")
video_embeddings = []
for video_path in video_files:
    embedding = get_video_embedding(video_path)
    if embedding is not None:  # Only add valid embeddings
        video_embeddings.append(embedding)

# Convert to a numpy array for KNN processing
if video_embeddings:  # Check if there are any valid embeddings
    video_embeddings = np.array(video_embeddings)
else:
    print("No valid video embeddings found. Exiting.")
    exit()

# Apply KNN to find similar videos based on cosine distance
print("Applying KNN to find duplicates...")
knn = NearestNeighbors(n_neighbors=2, metric='cosine').fit(video_embeddings)
distances, indices = knn.kneighbors(video_embeddings)

# Dynamically calculate similarity threshold based on mean distance
mean_distance = np.mean(distances[:, 1])
similarity_threshold = mean_distance * 1.0  # Adjust threshold to 90% of mean distance for optimal uniqueness
print(f"Calculated similarity threshold: {similarity_threshold}")

# Set to track videos that have been identified as duplicates
duplicate_videos = set()
unique_video_paths = []

# Loop through each video and check for duplicates
for i, (dist, idx) in enumerate(zip(distances, indices)):
    if dist[1] < similarity_threshold and idx[1] not in duplicate_videos:
        # Mark the second video as a duplicate
        duplicate_videos.add(idx[1])
        print(f"Duplicate found: {os.path.basename(video_files[idx[1]])} - Skipping this video.")
    else:
        # Save the unique video without subfolder structure
        unique_video_name = os.path.basename(video_files[i])  # Just get the video file name
        unique_video_paths.append(os.path.join(unique_video_dir, unique_video_name))
        os.system(f'cp "{video_files[i]}" "{unique_video_dir}/{unique_video_name}"')
        print(f"Saved unique video: {unique_video_name}")

# Summary of results
print(f"\nTotal Videos Processed: {len(video_files)}")
print(f"Unique Videos Kept: {len(unique_video_paths)}")
print(f"Duplicates Removed: {len(duplicate_videos)}")


In [ ]:
!pip install kaggle

In [4]:
# Move the API key to the correct location
!mkdir -p ~/.kaggle
!cp /kaggle/input/jsonkag/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json  # Ensure the API key file has the right permissions

In [ ]:
import os

# Define paths and metadata
dataset_name = "unique-video-crafts"  # Give your dataset a unique name
folder_to_upload = "/kaggle/working/unique_videos-crafts"  # Path to your dataset
metadata_file_path = f"{folder_to_upload}/dataset-metadata.json"  # Full path to metadata file

# Create the folder to upload if it doesn't exist
!mkdir -p $folder_to_upload

# Check if the metadata file exists
if not os.path.exists(metadata_file_path):
    # Create a metadata file for the dataset
    with open(metadata_file_path, 'w') as f:
        f.write('{\n'
                '  "title": "Unique-crafts-Videos",\n'
                '  "id": "tamimshadman/unique-video-crafts",\n'
                '  "licenses": [{"name": "CC0-1.0"}]\n'
                '}')

# Check if the metadata file was created successfully
!ls {folder_to_upload}

# Create a new dataset using the Kaggle API with --dir-mode option (using zip)
!kaggle datasets create -p $folder_to_upload --dir-mode zip
